In [83]:
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'TSLA', 'NVDA', 'JPM', 'V', 'WMT',
           'JNJ', 'PG', 'UNH', 'HD', 'DIS', 'MA', 'BAC', 'NFLX', 'ADBE', 'CRM']  # 20 акций
env = StockTradingEnv(
    tickers=tickers,
    start_date='2021-01-01',
    end_date='2025-12-31',
    initial_capital=1_000_000,
    lookback_window=60,
    transaction_cost_pct=0.002,
    slippage_factor=0.001,
    max_position_pct=0.1,
    min_signal_threshold=0.05,
    max_steps_per_episode=400,
    use_short=False
)

/var/folders/48/wvfw_hqj02x66nvb7tyts8bw0000gn/T/ipykernel_29439/3579676224.py:142: DeprecationWarning: Passing more than 2 positional arguments to np.maximum and np.minimum is deprecated. If you meant to use the third argument as an output, use the `out` keyword argument instead. If you hoped to work with more than 2 inputs, combine them into a single array and get the extrema for the relevant axis.
  tr = np.maximum(high - low,


In [51]:

obs, info = env.reset()
for _ in range(1000):
    # Здесь модель RWKV выдаст direction и size
    dummy_action = {
        'direction': np.random.uniform(-1, 1, size=20),
        'size': np.random.uniform(0, 1, size=20)
    }
    obs, reward, terminated, truncated, info = env.step(dummy_action)
    print(f"Reward: {reward}, Portfolio: {info['portfolio_value']:.2f}, Capital: {info['capital']:.2f}, Total_cost: {info['total_cost']:.2f}")

KeyError: slice(None, 20, None)

In [52]:
obs, _ = env.reset()
print("RESET obs shape:", obs.shape, "expected:", env.observation_space.shape)
print("Contains NaN:", np.isnan(obs).any(), "Inf:", np.isinf(obs).any())

action = env.action_space.sample()
obs2, reward, done, trunc, info = env.step(action)
print("STEP obs shape:", obs2.shape, "expected:", env.observation_space.shape)
print("Contains NaN:", np.isnan(obs2).any(), "Inf:", np.isinf(obs2).any())

RESET obs shape: (60, 8, 20) expected: (60, 8, 20)
Contains NaN: False Inf: False
STEP obs shape: (60, 8, 20) expected: (60, 8, 20)
Contains NaN: False Inf: False


In [53]:
from gymnasium import spaces
from stable_baselines3.common.env_checker import check_env

# Проверяем, что окружение соответствует API SB3
check_env(env) # убеждаемся, что всё в порядке

/Users/daniilogorodnikov/PycharmProjects/HFT-traiding/.venv/lib/python3.13/site-packages/stable_baselines3/common/env_checker.py:67: UserWarning: It seems that your observation  is an image but its `dtype` is (float32) whereas it has to be `np.uint8`. If your observation is not an image, we recommend you to flatten the observation to have only a 1D vector
  warnings.warn(
/Users/daniilogorodnikov/PycharmProjects/HFT-traiding/.venv/lib/python3.13/site-packages/stable_baselines3/common/env_checker.py:75: UserWarning: It seems that your observation space  is an image but the upper and lower bounds are not in [0, 255]. Because the CNN policy normalize automatically the observation you may encounter issue if the values are not in that range.
  warnings.warn(
/Users/daniilogorodnikov/PycharmProjects/HFT-traiding/.venv/lib/python3.13/site-packages/stable_baselines3/common/preprocessing.py:22: UserWarning: Treating image space as channels-last, while second dimension was smallest of the three.

In [66]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env

model = PPO(
    policy='MlpPolicy',
    env=env,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    learning_rate=3e-4,
    verbose=1
)

# Обучение
model.learn(total_timesteps=100000)
model.save("ppo_stock_trader")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 224      |
|    ep_rew_mean     | -3.8e-05 |
| time/              |          |
|    fps             | 748      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 237        |
|    ep_rew_mean          | -4.21e-05  |
| time/                   |            |
|    fps                  | 615        |
|    iterations           | 2          |
|    time_elapsed         | 6          |
|    total_timesteps      | 4096       |
| train/                  |            |
|    approx_kl            | 0.08383168 |
|    clip_fraction        | 0.55       |
|    clip_range           | 0.2        |
|    entropy_loss         | -56.7

In [84]:
from stable_baselines3 import PPO  # или SAC, TD3
import numpy as np

# Загрузка модели
model = PPO.load("ppo_stock_trader")

# Сброс среды
obs, info = env.reset()
done = False
total_reward = 0

while not done:
    # Предсказание действия (детерминированное)
    action, _ = model.predict(obs, deterministic=True)
    # action — это numpy-массив shape (2*n_assets,)

    # Шаг в среде
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    done = terminated or truncated

    # Логирование
    print(f"Reward: {reward:.6f}, Portfolio: {info['portfolio_value']:.2f}, "
          f"Capital: {info['capital']:.2f}, Cost: {info['total_cost']:.2f}")

print(f"Total reward: {total_reward:.4f}, Final portfolio: {info['portfolio_value']:.2f}")

Reward: 0.000000, Portfolio: 1000000.00, Capital: 1000000.00, Cost: 0.00
Reward: 0.000001, Portfolio: 1007580.80, Capital: 297895.98, Cost: 2104.01
Reward: -0.000000, Portfolio: 1004300.54, Capital: 581783.86, Cost: 2633.08
Reward: 0.000000, Portfolio: 1006174.23, Capital: 339131.34, Cost: 2073.31
Reward: 0.000000, Portfolio: 1007259.44, Capital: 483353.49, Cost: 2459.94
Reward: 0.000000, Portfolio: 1009452.38, Capital: 551490.50, Cost: 2213.17
Reward: -0.000000, Portfolio: 1007454.20, Capital: 651494.10, Cost: 937.63
Reward: 0.000000, Portfolio: 1008163.05, Capital: 538237.00, Cost: 1876.62
Reward: -0.000001, Portfolio: 998695.91, Capital: 298368.35, Cost: 2487.83
Reward: 0.000001, Portfolio: 1004283.39, Capital: 523555.82, Cost: 1193.03
Reward: -0.000000, Portfolio: 1004178.76, Capital: 332727.76, Cost: 1200.16
Reward: -0.000000, Portfolio: 999459.57, Capital: 223931.22, Cost: 1656.84
Reward: -0.000000, Portfolio: 997905.47, Capital: 523235.40, Cost: 1628.99
Reward: 0.000000, Portfol

In [82]:
import numpy as np
from stable_baselines3 import PPO

# Загрузка модели и среды
model = PPO.load("ppo_stock_trader")
obs, info = env.reset()

# Параметры
tickers = env.tickers
max_position_pct = env.max_position_pct  # 0.05
min_signal = 0.3  # минимальный модуль сигнала для действия

print("=" * 60)
print("Торговые сигналы модели:")
print("=" * 60)

step = 0
while True:
    action_flat, _ = model.predict(obs, deterministic=True)
    direction = action_flat[:env.n_assets]
    size = action_flat[env.n_assets:]

    # Шаг среды (чтобы получить info с ценами и капиталом)
    obs, reward, terminated, truncated, info = env.step(action_flat)
    holdings = info['holdings']  # массив текущего количества акций
    capital = info['capital']
    executed_shares = info['executed_shares']
    prices = env._get_current_prices()
    step += 1

    # --- Генерация читаемого отчёта ---
    print(f"\n--- Шаг {step} ---")
    current_prices = env._get_current_prices()
    capital = info['capital']
    portfolio_value = info['portfolio_value']

    print(f"Капитал: {capital:.2f} $, Портфель: {portfolio_value:.2f} $")
    print(holdings)
    print(executed_shares)

    if terminated or truncated:
        break

print("\n" + "=" * 60)
print(f"Торговля завершена. Итоговая стоимость портфеля: {info['portfolio_value']:.2f} $")

Торговые сигналы модели:

--- Шаг 1 ---
Капитал: 1000000.00 $, Портфель: 1000000.00 $
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

--- Шаг 2 ---
Капитал: 297895.98 $, Портфель: 1007580.80 $
[   0.            0.            0.            0.            0.
  453.34342484    0.            0.          461.12699995    0.
    0.          744.71255883    0.            0.          529.18453111
  275.25462387 2532.28658495    0.            0.          457.20556388]
[   0.            0.            0.            0.           -0.
  453.34342484    0.           -0.          461.12699995    0.
    0.          744.71255883    0.            0.          529.18453111
  275.25462387 2532.28658495   -0.           -0.          457.20556388]

--- Шаг 3 ---
Капитал: 581783.86 $, Портфель: 1004300.54 $
[ 334.86858954  169.26950132    0.          260.06068915  136.4797366
    0.            0.            0.            0.            0.

In [73]:
info

{'portfolio_value': np.float64(836025.3669042777),
 'capital': np.float64(182843.66675472478),
 'holdings': array([0.00000000e+00, 0.00000000e+00, 4.77013114e+02, 5.68434189e-14,
        3.12715494e+02, 0.00000000e+00, 0.00000000e+00, 5.09142848e+02,
        0.00000000e+00, 1.71397802e+03, 0.00000000e+00, 3.98756439e+02,
        1.64467629e+02, 2.64758214e+02, 3.29536312e+02, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 3.80118091e+02]),
 'total_cost': np.float64(1323.4589890015263),
 'commission': np.float64(881.7945425149193),
 'slippage': np.float64(441.6644464866071),
 'step': 312}